# Yelp Sentiment Analysis using a Transformer Built from Scratch

This notebook implements a Transformer encoder for sentiment classification **without using any pretrained model**.

In [16]:

import re
import math
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cpu


In [17]:

# Load dataset

df = pd.read_csv("/Users/sakshamchauhan/Desktop/GenAI-Restaurant-Review/Datasets/Yelp.csv")

df = df[["text", "sentiment"]].dropna()

print(df.head())
print(df["sentiment"].value_counts())


                                                text sentiment
0  Had a very lackluster experience today. All fo...  Negative
1  I know... I know... it's Phoenix but I expecte...  Negative
2  This should be called a speakeasy coffee shop....  Positive
3  With mixed word of mouth, I figured I'd judge ...   Neutral
4  It was an impromptu stop on our way to the the...  Negative
sentiment
Negative    3500
Positive    3500
Neutral     3000
Name: count, dtype: int64


In [18]:

# Encode labels

encoder = LabelEncoder()

df["label"] = encoder.fit_transform(df["sentiment"])

label_mapping = dict(
    zip(encoder.classes_, encoder.transform(encoder.classes_))
)

print(label_mapping)


{'Negative': 0, 'Neutral': 1, 'Positive': 2}


In [19]:

def clean_text(text):

    text = text.lower()

    text = re.sub(r"[^a-zA-Z ]", " ", text)

    text = re.sub(r"\s+", " ", text).strip()

    return text


df["clean_text"] = df["text"].apply(clean_text)


In [20]:

# Build vocabulary from scratch

MAX_VOCAB = 20000
MAX_LEN = 128

word_freq = {}

for sentence in df["clean_text"]:

    for word in sentence.split():

        word_freq[word] = word_freq.get(word, 0) + 1

sorted_words = sorted(
    word_freq.items(),
    key=lambda x: x[1],
    reverse=True
)

vocab = {
    "<PAD>": 0,
    "<UNK>": 1
}

for word, _ in sorted_words[: MAX_VOCAB - 2]:

    vocab[word] = len(vocab)

print("Vocabulary size:", len(vocab))


Vocabulary size: 20000


In [21]:

def encode_text(sentence):

    tokens = []

    for word in sentence.split():

        tokens.append(vocab.get(word, vocab["<UNK>"]))

    tokens = tokens[:MAX_LEN]

    padding = MAX_LEN - len(tokens)

    tokens += [0] * padding

    return tokens


df["tokens"] = df["clean_text"].apply(encode_text)


In [22]:

X_train, X_test, y_train, y_test = train_test_split(
    df["tokens"].tolist(),
    df["label"].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)


In [23]:

class YelpDataset(Dataset):

    def __init__(self, X, y):

        self.X = X
        self.y = y

    def __len__(self):

        return len(self.X)

    def __getitem__(self, idx):

        return (
            torch.tensor(self.X[idx], dtype=torch.long),
            torch.tensor(self.y[idx], dtype=torch.long)
        )


train_dataset = YelpDataset(X_train, y_train)
test_dataset = YelpDataset(X_test, y_test)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64
)


In [24]:

class PositionalEncoding(nn.Module):

    def __init__(self, d_model, max_len=5000):

        super().__init__()

        pe = torch.zeros(max_len, d_model)

        position = torch.arange(
            0, max_len
        ).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(
                0,
                d_model,
                2
            ) * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer(
            "pe",
            pe.unsqueeze(0)
        )

    def forward(self, x):

        return x + self.pe[:, :x.size(1)]


In [25]:

class TransformerClassifier(nn.Module):

    def __init__(
        self,
        vocab_size,
        num_classes,
        d_model=128,
        n_heads=4,
        n_layers=2,
        hidden_dim=256,
        dropout=0.2
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            d_model,
            padding_idx=0
        )

        self.positional_encoding = PositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=hidden_dim,
            dropout=dropout,
            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=n_layers
        )

        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(
            d_model,
            num_classes
        )

    def forward(self, x):

        x = self.embedding(x)

        x = self.positional_encoding(x)

        x = self.encoder(x)

        x = x.mean(dim=1)

        x = self.dropout(x)

        return self.fc(x)


model = TransformerClassifier(
    vocab_size=len(vocab),
    num_classes=len(label_mapping)
).to(device)

print(model)


TransformerClassifier(
  (embedding): Embedding(20000, 128, padding_idx=0)
  (positional_encoding): PositionalEncoding()
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=256, bias=True)
        (dropout): Dropout(p=0.2, inplace=False)
        (linear2): Linear(in_features=256, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.2, inplace=False)
        (dropout2): Dropout(p=0.2, inplace=False)
      )
    )
  )
  (dropout): Dropout(p=0.2, inplace=False)
  (fc): Linear(in_features=128, out_features=3, bias=True)
)


In [26]:

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.0005
)


In [27]:

EPOCHS = 12

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    for batch_x, batch_y in train_loader:

        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()

        outputs = model(batch_x)

        loss = criterion(
            outputs,
            batch_y
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch + 1}/{EPOCHS}, "
        f"Loss: {total_loss / len(train_loader):.4f}"
    )


Epoch 1/12, Loss: 0.9113
Epoch 2/12, Loss: 0.6506
Epoch 3/12, Loss: 0.5520
Epoch 4/12, Loss: 0.4866
Epoch 5/12, Loss: 0.4330
Epoch 6/12, Loss: 0.3823
Epoch 7/12, Loss: 0.3329
Epoch 8/12, Loss: 0.2648
Epoch 9/12, Loss: 0.2183
Epoch 10/12, Loss: 0.1677
Epoch 11/12, Loss: 0.1357
Epoch 12/12, Loss: 0.1116


In [28]:

model.eval()

predictions = []
actual = []

with torch.no_grad():

    for batch_x, batch_y in test_loader:

        batch_x = batch_x.to(device)

        outputs = model(batch_x)

        preds = torch.argmax(
            outputs,
            dim=1
        )

        predictions.extend(
            preds.cpu().numpy()
        )

        actual.extend(
            batch_y.numpy()
        )

print(
    "Accuracy:",
    accuracy_score(
        actual,
        predictions
    )
)

print(
    classification_report(
        actual,
        predictions,
        target_names=encoder.classes_
    )
)


Accuracy: 0.7635
              precision    recall  f1-score   support

    Negative       0.85      0.76      0.80       700
     Neutral       0.75      0.64      0.69       600
    Positive       0.71      0.88      0.78       700

    accuracy                           0.76      2000
   macro avg       0.77      0.76      0.76      2000
weighted avg       0.77      0.76      0.76      2000



In [29]:

def predict_sentiment(text):

    model.eval()

    text = clean_text(text)

    tokens = encode_text(text)

    tensor = torch.tensor(
        [tokens],
        dtype=torch.long
    ).to(device)

    with torch.no_grad():

        output = model(tensor)

        pred = torch.argmax(
            output,
            dim=1
        ).item()

    return encoder.inverse_transform([pred])[0]


review = "The food was average and the staff was really normal"

print(
    predict_sentiment(review)
)


Positive


In [30]:
import os
import pickle
import torch

# Create Models folder if it doesn't exist
os.makedirs("Models", exist_ok=True)

# Save model weights
torch.save(
    model.state_dict(),
    "Models/transformer_model.pth"
)

# Save vocabulary
with open("Models/vocab.pkl", "wb") as f:
    pickle.dump(vocab, f)

# Save label encoder
with open("Models/label_encoder.pkl", "wb") as f:
    pickle.dump(encoder, f)

# Save model configuration
config = {
    "max_len": 128,
    "vocab_size": len(vocab),
    "d_model": 128,
    "n_heads": 4,
    "n_layers": 2,
    "hidden_dim": 256,
    "dropout": 0.2,
    "num_classes": len(label_mapping)
}

with open("Models/config.pkl", "wb") as f:
    pickle.dump(config, f)

print("\nFiles saved successfully:")
print("Models/transformer_model.pth")
print("Models/vocab.pkl")
print("Models/label_encoder.pkl")
print("Models/config.pkl")


Files saved successfully:
Models/transformer_model.pth
Models/vocab.pkl
Models/label_encoder.pkl
Models/config.pkl
